# ДЗ1. Дополнительная задача: детекция искусственно сгенерированных текстов

В этом ноутбуке решается задача определения, является ли текст машинно-сгенерированным или написанным человеком.

Работа построена как базовый анализ: сначала мы смотрим на структуру датасета и простые признаки текста, затем обучаем две базовые модели и отдельно фиксируем важное ограничение набора данных, связанное с неразмеченным `test`-сплитом.

Используем корпус `CoAT`, на который указывает задание, и строим два уровня решения:
- анализ и базовую модель на простых лексических признаках;
- более сильную базовую модель на `TF-IDF` признаках.


## 1. Что нужно для запуска

Для загрузки `CoAT` нужен пакет `datasets`. Если он еще не установлен, его можно поставить командой:

`python -m pip install datasets`

Дальше ноутбук использует подмножество `binary`, потому что именно оно соответствует бинарной задаче: человек или машина.


In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

try:
    from datasets import load_dataset
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("Нужно установить пакет datasets: python -m pip install datasets") from e

plt.style.use("ggplot")
pd.set_option("display.max_colwidth", 200)


## 2. Загрузка датасета CoAT

Загружаем именно бинарную постановку, где требуется отличать тексты, написанные человеком, от машинно-сгенерированных текстов.


In [ ]:
dataset = load_dataset("RussianNLP/coat", "binary")
dataset


In [ ]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas() if "validation" in dataset else dataset["val"].to_pandas()
test_df = dataset["test"].to_pandas()
has_test_labels = sorted(test_df["label"].dropna().unique().tolist()) != [-1]

train_df.head()


## 3. Первичный анализ датасета

Сначала смотрим на размеры сплитов, распределение классов и длины текстов. Это нужно, чтобы понять, насколько задача сбалансирована и какие простые признаки могут быть полезны.


In [ ]:
splits_stats_df = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df)},
        {"split": "val", "rows": len(val_df)},
        {"split": "test", "rows": len(test_df)},
    ]
)
splits_stats_df


In [ ]:
train_df["label"].value_counts()


In [ ]:
plt.figure(figsize=(7, 5))
train_df["label"].value_counts().plot(kind="bar", color=["#4C78A8", "#E45756"])
plt.title("Распределение классов в train")
plt.xlabel("Класс")
plt.ylabel("Число текстов")
plt.tight_layout()
plt.show()


In [ ]:
train_df["char_len"] = train_df["text"].str.len()
train_df["word_len"] = train_df["text"].str.split().str.len()

train_df[["char_len", "word_len"]].describe()


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(train_df["word_len"], bins=50, color="#72B7B2")
plt.title("Распределение длины текстов в словах")
plt.xlabel("Число слов")
plt.ylabel("Число текстов")
plt.tight_layout()
plt.show()


## 4. Простые лексические признаки

Здесь используем идеи из основной части ДЗ: длину текста, среднюю длину слова, разнообразие словаря и долю повторов. Это слабее, чем полноценные n-gram признаки, но хорошо показывает, сколько можно выжать из базовой статистики коллекции.


In [ ]:
TOKEN_PATTERN = re.compile(r"[а-яёa-z]+", flags=re.IGNORECASE)

def tokenize_text(text):
    return [token.lower() for token in TOKEN_PATTERN.findall(text)]


def extract_lexical_features(text):
    tokens = tokenize_text(text)
    token_count = len(tokens)
    unique_count = len(set(tokens))
    avg_token_length = np.mean([len(token) for token in tokens]) if tokens else 0
    ttr = unique_count / token_count if token_count else 0
    hapax_ratio = sum(1 for _, count in Counter(tokens).items() if count == 1) / unique_count if unique_count else 0
    digit_share = sum(char.isdigit() for char in text) / len(text) if text else 0
    return {
        "token_count": token_count,
        "unique_count": unique_count,
        "avg_token_length": avg_token_length,
        "type_token_ratio": ttr,
        "hapax_ratio": hapax_ratio,
        "digit_share": digit_share,
    }


In [ ]:
train_features_df = pd.DataFrame([extract_lexical_features(text) for text in train_df["text"]])
val_features_df = pd.DataFrame([extract_lexical_features(text) for text in val_df["text"]])
test_features_df = pd.DataFrame([extract_lexical_features(text) for text in test_df["text"]])

train_features_df.head()


Перед обучением полезно посмотреть, отличаются ли классы хотя бы по самым простым признакам.


In [ ]:
feature_preview_df = pd.concat([train_df[["label"]], train_features_df], axis=1)
feature_preview_df.groupby("label").mean(numeric_only=True)


In [ ]:
plt.figure(figsize=(8, 5))
for label, color in [(feature_preview_df['label'].unique()[0], '#4C78A8'), (feature_preview_df['label'].unique()[1], '#E45756')]:
    subset = feature_preview_df[feature_preview_df['label'] == label]
    plt.hist(subset['type_token_ratio'], bins=50, alpha=0.5, label=str(label), color=color)
plt.title("Сравнение type-token ratio по классам")
plt.xlabel("Type-token ratio")
plt.ylabel("Число текстов")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Базовая модель на простых признаках

Сначала обучаем самую простую модель на агрегированных характеристиках текста. Такая базовая модель нужна, чтобы понять, насколько далеко можно уйти вообще без учета конкретных слов и биграмм.


In [ ]:
lex_clf = LogisticRegression(max_iter=1000, random_state=42)
lex_clf.fit(train_features_df, train_df["label"])

val_pred_lex = lex_clf.predict(val_features_df)
test_pred_lex = lex_clf.predict(test_features_df)

print("Validation accuracy:", round(accuracy_score(val_df["label"], val_pred_lex), 4))
print("Validation macro F1:", round(f1_score(val_df["label"], val_pred_lex, average="macro"), 4))


In [ ]:
print(classification_report(val_df["label"], val_pred_lex))


## 6. Базовая модель на TF-IDF

Теперь переходим к частотному представлению текста. Это уже более содержательная базовая модель, потому что она видит сами слова и их сочетания, а не только агрегированные статистики.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_features=50000,
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["text"])
X_val_tfidf = tfidf_vectorizer.transform(val_df["text"])
X_test_tfidf = tfidf_vectorizer.transform(test_df["text"])


In [ ]:
tfidf_clf = LogisticRegression(max_iter=1000, random_state=42)
tfidf_clf.fit(X_train_tfidf, train_df["label"])

val_pred_tfidf = tfidf_clf.predict(X_val_tfidf)
test_pred_tfidf = tfidf_clf.predict(X_test_tfidf)

print("Validation accuracy:", round(accuracy_score(val_df["label"], val_pred_tfidf), 4))
print("Validation macro F1:", round(f1_score(val_df["label"], val_pred_tfidf, average="macro"), 4))


In [ ]:
print(classification_report(val_df["label"], val_pred_tfidf))


## 7. Сравнение базовых моделей

Собираем результаты рядом, чтобы было видно, насколько простой набор статистик уступает модели на частотном представлении текста.


In [ ]:
results_rows = [
    {
        "Модель": "Лексические признаки",
        "Val accuracy": accuracy_score(val_df["label"], val_pred_lex),
        "Val macro F1": f1_score(val_df["label"], val_pred_lex, average="macro"),
    },
    {
        "Модель": "TF-IDF + LogisticRegression",
        "Val accuracy": accuracy_score(val_df["label"], val_pred_tfidf),
        "Val macro F1": f1_score(val_df["label"], val_pred_tfidf, average="macro"),
    },
]

if has_test_labels:
    results_rows[0]["Test accuracy"] = accuracy_score(test_df["label"], test_pred_lex)
    results_rows[0]["Test macro F1"] = f1_score(test_df["label"], test_pred_lex, average="macro")
    results_rows[1]["Test accuracy"] = accuracy_score(test_df["label"], test_pred_tfidf)
    results_rows[1]["Test macro F1"] = f1_score(test_df["label"], test_pred_tfidf, average="macro")
else:
    results_rows[0]["Test accuracy"] = np.nan
    results_rows[0]["Test macro F1"] = np.nan
    results_rows[1]["Test accuracy"] = np.nan
    results_rows[1]["Test macro F1"] = np.nan

results_df = pd.DataFrame(results_rows)
results_df


In [ ]:
metric_cols = ["Val macro F1"] + (["Test macro F1"] if has_test_labels else [])
plot_df = results_df.set_index("Модель")[metric_cols]
plot_df.plot(kind="bar", figsize=(8, 5), color=["#4C78A8"] if len(metric_cols) == 1 else ["#4C78A8", "#F58518"])
plt.title("Сравнение качества baseline-моделей" if has_test_labels else "Сравнение baseline-моделей на validation")
plt.ylabel("Macro F1")
plt.xticks(rotation=10)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## 8. Предсказания на неразмеченном `test`

У опубликованного `test`-сплита в конфигурации `binary` нет истинных меток: там в колонке `label` стоит `-1`. Поэтому ниже мы не считаем `accuracy` и `F1` на `test`, а только смотрим, как модели распределяют свои предсказания по классам.


In [ ]:
test_prediction_share_df = pd.DataFrame(
    [
        {
            "Модель": "Лексические признаки",
            "Доля предсказаний класса 0": (test_pred_lex == 0).mean(),
            "Доля предсказаний класса 1": (test_pred_lex == 1).mean(),
        },
        {
            "Модель": "TF-IDF + LogisticRegression",
            "Доля предсказаний класса 0": (test_pred_tfidf == 0).mean(),
            "Доля предсказаний класса 1": (test_pred_tfidf == 1).mean(),
        },
    ]
)
test_prediction_share_df


In [ ]:
plot_test_df = test_prediction_share_df.set_index("Модель")
plot_test_df.plot(kind="bar", figsize=(8, 5), color=["#4C78A8", "#F58518"])
plt.title("Распределение предсказаний на unlabeled test")
plt.ylabel("Доля")
plt.xticks(rotation=10)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## 9. Итоговые выводы

Датасет `CoAT binary` в этой конфигурации хорошо сбалансирован: в `train` по `86 199` объектов каждого класса, а в `validation` по `12 314`. По простым лексическим признакам baseline дает `Val macro F1 = 0.6203`, а модель `TF-IDF + LogisticRegression` заметно сильнее и достигает `0.7157`.

Это хороший практический вывод для ДЗ: агрегированные статистики текста действительно несут сигнал, но они недостаточны как сильное решение. Частотное представление слов и биграмм уже заметно лучше улавливает различие между классами.

Отдельно важно отметить особенность датасета: в `test` у всех объектов `label = -1`, поэтому полноценные test-метрики здесь считать нельзя. Вместо этого можно анализировать только распределение предсказаний. У лексической модели доля предсказаний класса `1` на `test` равна примерно `0.4426`, у `TF-IDF`-модели примерно `0.4716`.

По средним лексическим признакам классы тоже различаются: у класса `1` тексты в среднем длиннее и показывают более высокие `unique_count`, `type_token_ratio` и `hapax_ratio`. Это подтверждает, что даже простая статистика текста уже дает полезные сигналы для детекции, но лучшая baseline-модель все равно опирается на само содержимое текста, а не только на агрегаты.
